# 06 · Functional programming · Worked solutions

Try the exercises in the student chapter first. This notebook is self-contained: run it from
a fresh kernel in order. Each exercise explains one implementation and then tests both the
normal case and cases that reveal common mistakes. Assertions here are checks of our own
code; required user-input validation is implemented with explicit exceptions.


## Exercise 1 · Lazy text transformations

`map` transforms each requested string; `filter(None, ...)` drops the resulting empty
strings. Consuming the iterator does not modify the source list. A second traversal of the
same iterator is empty; create a new pipeline if you need to repeat the transformation.


In [ ]:
def cleaned_words(words):
    """Lazily strip, lowercase and keep nonempty text values."""
    return filter(None, map(lambda word: word.strip().lower(), words))

words = [" Python ", "", "  ", " DATA "]
original = words.copy()
pipeline = cleaned_words(words)
assert iter(pipeline) is pipeline
assert list(pipeline) == ["python", "data"]
assert list(pipeline) == []
assert words == original
stripped = [word.strip().lower() for word in words]
assert [word for word in stripped if word] == list(cleaned_words(words))
assert list(cleaned_words([])) == []

requested = []
def text_source():
    for word in [" FIRST ", " SECOND "]:
        requested.append(word)
        yield word

lazy = cleaned_words(text_source())
assert requested == []
assert next(lazy) == "first"
assert requested == [" FIRST "]
print(list(cleaned_words(words)))


## Exercise 2 · Predicates and sorting keys

Testing divisors through the square root also catches perfect squares. `sorted` returns a
new list, and a tuple key compares the descending score first, then the ascending name.
Negating the score reverses only the numeric part of the ordering.


In [ ]:
def is_prime(number):
    """Return whether an integer is prime; numbers below two are not prime."""
    if number < 2:
        return False
    divisor = 2
    while divisor * divisor <= number:
        if number % divisor == 0:
            return False
        divisor += 1
    return True

for number in [-7, 0, 1, 4, 49, 100]:
    assert not is_prime(number)
for number in [2, 3, 29, 97]:
    assert is_prime(number)
expected = [2, 3, 5, 7, 11, 13, 17, 19, 23, 29]
assert list(filter(is_prime, range(30))) == expected
assert [number for number in range(30) if is_prime(number)] == expected
assert list(filter(is_prime, [])) == []
records = [("Ada", 8), ("Lin", 5), ("Bo", 8)]
original = records.copy()
ranked = sorted(records, key=lambda record: (-record[1], record[0]))
assert ranked == [("Ada", 8), ("Bo", 8), ("Lin", 5)]
assert records == original
print(expected, ranked)


## Exercise 3 · Bound the consumer

The generator can produce as many values as needed; `islice` imposes the exact finite
count. Validation is in a normal wrapper function so invalid counts fail immediately.
Each call to `fibonacci_numbers()` creates independent state.


In [ ]:
from itertools import islice

def fibonacci_numbers():
    """Yield the infinite stream 1, 1, 2, 3, 5, ...; consume it with a bound."""
    a, b = 0, 1
    while True:
        a, b = b, a + b
        yield a

def first_fibonacci(count):
    if isinstance(count, bool) or not isinstance(count, int):
        raise TypeError("count must be an integer")
    if count < 0:
        raise ValueError("count must not be negative")
    return list(islice(fibonacci_numbers(), count))

assert first_fibonacci(0) == []
assert first_fibonacci(1) == [1]
assert first_fibonacci(8) == [1, 1, 2, 3, 5, 8, 13, 21]
for count, expected in [(-1, ValueError), (2.5, TypeError), (True, TypeError)]:
    try:
        first_fibonacci(count)
    except expected:
        pass
    else:
        raise AssertionError(f"Expected {expected.__name__}")
left, right = fibonacci_numbers(), fibonacci_numbers()
assert list(islice(left, 4)) == [1, 1, 2, 3]
assert next(right) == 1
assert next(left) == 5
left.close()
right.close()
print(first_fibonacci(8))


## Exercise 4 · Retain a separate configuration

The outer call validates the divisor and creates an enclosing scope. The returned function
uses that scope later when a number is supplied. Calling the inner function inside the
factory would need a number immediately and would return a boolean instead of a predicate.
Zero is divisible by every nonzero integer; negative divisors work with the same test.


In [ ]:
def make_divisibility_test(divisor):
    """Return a predicate retaining its own nonzero integer divisor."""
    if isinstance(divisor, bool) or not isinstance(divisor, int):
        raise TypeError("divisor must be an integer")
    if divisor == 0:
        raise ValueError("divisor must not be zero")

    def divisible(number):
        return number % divisor == 0

    return divisible

by_three = make_divisibility_test(3)
by_five = make_divisibility_test(5)
by_negative_two = make_divisibility_test(-2)
assert callable(by_three)
assert by_three(0) and by_three(-6) and not by_three(10)
assert by_five(10) and not by_five(6)
assert by_negative_two(4) and by_negative_two(-4)
assert not by_negative_two(3)
assert list(filter(by_three, range(10))) == [0, 3, 6, 9]
assert by_three is not by_five
for divisor, expected in [(0, ValueError), ("3", TypeError), (True, TypeError)]:
    try:
        make_divisibility_test(divisor)
    except expected:
        pass
    else:
        raise AssertionError(f"Expected {expected.__name__}")
print(list(filter(by_three, range(10))))


## Exercise 5 · A counting decorator

The count belongs to the returned wrapper, so independently decorated functions have
independent counters. Incrementing before delegation includes failed attempts. There is no
`except` in the wrapper: the original exception reaches the caller unchanged. `wraps`
preserves the visible name, docstring, and a reference to the wrapped function.


In [ ]:
from functools import wraps

def count_calls(function):
    @wraps(function)
    def wrapper(*args, **kwargs):
        wrapper.calls += 1
        return function(*args, **kwargs)
    wrapper.calls = 0
    return wrapper

@count_calls
def adjusted(value, *, increment=1):
    """Add an increment to a nonnegative value."""
    if value < 0:
        raise ValueError("value must not be negative")
    return value + increment

@count_calls
def doubled(value):
    return value * 2

assert adjusted.calls == 0 and doubled.calls == 0
assert adjusted(3, increment=4) == 7
assert adjusted.calls == 1
assert adjusted.__name__ == "adjusted"
assert adjusted.__doc__ == "Add an increment to a nonnegative value."
try:
    adjusted(-1)
except ValueError as error:
    assert str(error) == "value must not be negative"
else:
    raise AssertionError("The original failure must propagate")
assert adjusted.calls == 2
assert doubled.calls == 0
assert doubled(5) == 10 and doubled.calls == 1
assert adjusted.__wrapped__(2, increment=3) == 5
assert adjusted.calls == 2  # A direct call bypassed the wrapper.
print("Attempted calls:", adjusted.calls, doubled.calls)


## Exercise 6 · Verify reuse through the cache

Repeated calls with the same argument reuse the result, so the second call adds one hit
and no misses. A current-time function should produce a new time on each call; caching
would instead reuse an old time. Memoization is appropriate only when that reuse matches
the function's intended behavior. These small indices keep recursive depth bounded.


In [ ]:
from functools import lru_cache

@lru_cache(maxsize=None, typed=True)
def fibonacci_at(index):
    """Return F(index) for a small nonnegative integer; F(0)=0, F(1)=1.

    Typed keys keep booleans and floats separate from validated integer keys.
    Caching avoids repeated recursive subproblems. Very large indices should
    use an iterative algorithm to avoid Python's recursion limit.
    """
    if isinstance(index, bool) or not isinstance(index, int):
        raise TypeError("index must be an integer")
    if index < 0:
        raise ValueError("index must not be negative")
    if index < 2:
        return index
    return fibonacci_at(index - 1) + fibonacci_at(index - 2)

fibonacci_at.cache_clear()
assert fibonacci_at(0) == 0
assert fibonacci_at(1) == 1
assert fibonacci_at(10) == 55
fibonacci_at.cache_clear()
result = fibonacci_at(20)
before = fibonacci_at.cache_info()
assert result == 6765
assert fibonacci_at(20) == result
after = fibonacci_at.cache_info()
assert after.hits == before.hits + 1
assert after.misses == before.misses
assert after.currsize == before.currsize
assert fibonacci_at(index=1) == 1
for index, expected in [(-1, ValueError), (2.5, TypeError), (True, TypeError), (1.0, TypeError)]:
    try:
        fibonacci_at(index=index)
    except expected:
        pass
    else:
        raise AssertionError(f"Expected {expected.__name__}")
print(result, after)
fibonacci_at.cache_clear()
assert fibonacci_at.cache_info().currsize == 0
